# Job Postings 2026: India and United States: starter notebook

This file is a **sample**. The full data, the withheld fields and monthly updates come from the [Radar API](https://github.com/fliarbi/radar-api-mcp). This notebook loads the sample, explores it, then gets one record's full detail from the API.

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
path = (glob.glob('/kaggle/input/**/job_postings_2026_sample.csv', recursive=True) or ['job_postings_2026_sample.csv'])[0]
df = pd.read_csv(path, low_memory=False)
print(df.shape); df.head()

In [ ]:
df.describe(include="all").T.head(30)

In [ ]:
df['month'] = pd.to_datetime(df.posted_date).dt.to_period('M')
df.pivot_table(index='month', columns='country', values='posting_id', aggfunc='count')

In [ ]:
sk = df.dropna(subset=['skills']).assign(skill=lambda d: d.skills.str.split('|')).explode('skill')
sk.skill.value_counts().head(25)

In [ ]:
s = df.dropna(subset=['salary_annual_usd'])
s.groupby(['country', 'role']).salary_annual_usd.median().unstack(0).dropna().sort_values('US', ascending=False).head(15)

## Get the full record from the Radar API

The sample has one row per posting without description text. The API searches the full, daily-updated postings with filters for skills, salary, experience and freshness. [Create a key](https://fliarbi.com/radar-api/developer), then add it to Kaggle **Add-ons → Secrets** as `RADAR_API_KEY`.

In [ ]:
import os, json, requests
key = os.environ.get('RADAR_API_KEY')
try:
    from kaggle_secrets import UserSecretsClient
    key = key or UserSecretsClient().get_secret('RADAR_API_KEY')
except Exception:
    pass
value = df['company'].dropna().astype(str).iloc[0]
if key:
    r = requests.post('https://radar-api.fliarbi.com/v1/jobs/search', headers={'Authorization': f'Bearer {key}'},
                      json={'company': value}, timeout=60)
    print(json.dumps(r.json(), indent=1)[:3000])
else:
    print('Set RADAR_API_KEY to fetch the full record for', value)

Each row also has `fliar_profile_url`, the company's full profile. Repository, docs and MCP setup: https://github.com/fliarbi/radar-api-mcp